<a href="https://colab.research.google.com/github/Afridi-7/confidence-competence/blob/main/01_apacc_cell_counts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Counting the APACC cells per smear



## Step 1 — Connect Google Drive and set up


In [1]:
import os, re, json, time, zipfile, random, logging
import numpy as np, pandas as pd, requests

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/CINC/data'
except ImportError:                      # lets the notebook run outside Colab for testing
    OUT = './CINC/data'
os.makedirs(OUT, exist_ok=True)

SEED = 20261016
random.seed(SEED); np.random.seed(SEED)
pd.set_option('display.max_colwidth', 200)   # show full file paths when printing
logging.basicConfig(filename=os.path.join(OUT, 'count_log.txt'), level=logging.INFO,
                    format='%(asctime)s %(message)s', force=True)
def log(m):
    print(m); logging.info(m)

API, NODE = 'https://api.osf.io/v2', 'fp2xe'
CLASSES = ('healthy', 'unhealthy', 'bothcells', 'rubbish')
# Numbers published in the APACC paper (Kupas et al., Sci. Data 2024) -- we check against them
PAPER = dict(total=103675, healthy=34721, unhealthy=2942, bothcells=3884,
             rubbish=62074, smears=107, test_smears=20)
log(f'Saving outputs to {OUT}')

Mounted at /content/drive
Saving outputs to /content/drive/MyDrive/CINC/data


## Step 2 — List every file on OSF

In [2]:
S = requests.Session()

def get_json(url, params=None, tries=7):
    for k in range(tries):
        r = S.get(url, params=params, timeout=90)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503, 504):     # server busy -> wait and retry
            time.sleep(min(2 ** k, 60)); continue
        r.raise_for_status()
    raise RuntimeError(f'OSF request kept failing: {url}')

def paged(url):
    params = {'page[size]': 100}
    while url:
        j = get_json(url, params); params = None
        yield from j['data']
        url = j['links'].get('next')

def walk(folder_url, rows):
    for it in paged(folder_url):
        a = it['attributes']
        if a['kind'] == 'folder':
            walk(it['relationships']['files']['links']['related']['href'], rows)
        else:
            rows.append(dict(path=a['materialized_path'], name=a['name'],
                             size=a.get('size') or 0, download=it['links'].get('download')))
            if len(rows) % 5000 == 0:
                print(f'  ... {len(rows):,} files listed')

def all_nodes(node):                      # the project may keep data in sub-projects
    ids = [node]
    for ch in paged(f'{API}/nodes/{node}/children/'):
        ids += all_nodes(ch['id'])
    return ids

listing_csv = os.path.join(OUT, 'osf_listing.csv')
if os.path.exists(listing_csv):
    listing = pd.read_csv(listing_csv)
    log(f'Loaded cached OSF listing: {len(listing):,} files')
else:
    rows, t0 = [], time.time()
    for nid in all_nodes(NODE):
        n0 = len(rows)
        walk(f'{API}/nodes/{nid}/files/osfstorage/', rows)
        log(f'node {nid}: {len(rows) - n0:,} files')
    listing = pd.DataFrame(rows)
    listing.to_csv(listing_csv, index=False)
    log(f'OSF listing done: {len(listing):,} files in {(time.time() - t0) / 60:.1f} min')

# Look at what we got: top-level folders and a few example paths
top = listing['path'].str.split('/').str[1].value_counts()
print('\nTop-level folders on OSF (folder: number of files):'); print(top.to_string())
print('\nExample paths:'); print(listing['path'].head(5).to_string(index=False))

Loaded cached OSF listing: 148,482 files

Top-level folders on OSF (folder: number of files):
path
dataset_full      147282
dataset_sample      1200

Example paths:
 /dataset_sample/cell-images/test/bothcells/3255_18_206000_198000__2__877_792__2021-07-07.png
 /dataset_sample/cell-images/test/bothcells/3255_18_220000_30000__2__387_1095__2021-07-07.png
/dataset_sample/cell-images/test/bothcells/3255_18_220000_30000__8__1212_1340__2021-07-07.png
  /dataset_sample/cell-images/test/bothcells/3255_18_220000_32000__5__563_445__2021-07-07.png
  /dataset_sample/cell-images/test/bothcells/3255_18_220000_32000__3__423_543__2021-07-07.png


## Step 3 — Turn file names into a table (the "manifest")


In [4]:
CELL_ROOT = '/dataset_full/cell-images/'
cell_paths = listing.loc[listing['path'].str.startswith(CELL_ROOT)
                         & listing['path'].str.lower().str.endswith('.png'), 'path']
print(f'PNG files under {CELL_ROOT}: {len(cell_paths):,}')
print('Raw example file names (as stored on OSF):')
print(cell_paths.sample(min(8, len(cell_paths)), random_state=0).str.split('/').str[-1].to_string(index=False))

# Smear id = the first two number groups of the file name, e.g. "12916_17" (same form as the
# whole-slide folders on OSF, "669_15"). Works whether the rest uses "_" or "-".
SMEAR_RE = re.compile(r'^(?P<smear>\d+[_-]\d+)[_-]')

def parse(path):
    parts = path[len(CELL_ROOT):].split('/')          # e.g. ['train', 'rubbish_part2', '<file>.png']
    if len(parts) != 3:
        return dict(bad=path)
    split, folder, fname = parts
    cls = next((c for c in CLASSES if folder.lower().startswith(c)), None)
    m = SMEAR_RE.match(fname)
    if split not in ('train', 'test') or cls is None or m is None:
        return dict(bad=path)
    return dict(split=split, cls=cls, folder=folder, smear=m['smear'].replace('-', '_'),
                file=fname, path=path)

parsed = [parse(p) for p in cell_paths]
bad = [r['bad'] for r in parsed if 'bad' in r]
log(f'Cell images read: {len(parsed) - len(bad):,}   could not read: {len(bad):,}')
if bad:
    print('Unreadable examples (send these to me):'); print('\n'.join(bad[:10]))
if len(bad) == len(parsed):
    raise SystemExit('No cell image could be read. Send me the output above.')

man = pd.DataFrame([r for r in parsed if 'bad' not in r]).drop_duplicates('path')
print('\nImages per folder:')
print(man.groupby(['split', 'folder']).size().to_string())

# Cross-check: the 34 whole-slide folders on OSF are named by smear id; they should all be found
wsi = set(listing.loc[listing['path'].str.contains('/whole-slide-images-with-most-cells/'), 'path']
          .str.split('/').str[3])
print(f'\nWhole-slide folders whose smear id is in our manifest: {len(wsi & set(man.smear))} of {len(wsi)}')
print('Example smear IDs:', sorted(man['smear'].unique())[:8])

dup = int(man.duplicated('file', keep=False).sum())
if dup:
    log(f'!! {dup} cell files share a name across folders -- tell me before going on')
man.drop(columns=['folder']).to_csv(os.path.join(OUT, 'cells_manifest.csv'), index=False)
log(f'Manifest saved: {len(man):,} cell images')

PNG files under /dataset_full/cell-images/: 103,674
Raw example file names (as stored on OSF):
 28998_16_72000_196000__24__1450_688__2021-05-29.png
    1675_17_214000_44000__4__515_135__2021-06-21.png
 27088_16_60000_88000__38__1003_1380__2021-06-21.png
 3936_18_222000_140000__8__1725_1347__2021-05-29.png
 11218_15_70000_142000__41__1549_519__2021-06-21.png
 10358_18_176000_148000__2__820_1915__2021-05-29.png
 28998_16_96000_144000__28__1928_112__2021-05-29.png
19207_16_176000_120000__14__363_1809__2021-05-05.png
Cell images read: 103,674   could not read: 0

Images per folder:
split  folder       
test   bothcells          436
       healthy           5826
       rubbish          11757
       unhealthy          576
train  bothcells         3448
       healthy          28894
       rubbish          36001
       rubbish_part2    14370
       unhealthy         2366

Whole-slide folders whose smear id is in our manifest: 33 of 68
Example smear IDs: ['10030_18', '10052_18', '10175_13', '10

## Step 4 — Check our counts against the paper


In [5]:
cls_n = man['cls'].value_counts().to_dict()
smear_split = man.groupby('smear')['split'].nunique()
checks = {
    'total': (len(man), PAPER['total']),
    **{c: (cls_n.get(c, 0), PAPER[c]) for c in CLASSES},
    'smears': (man['smear'].nunique(), PAPER['smears']),
    'test_smears': (man.loc[man.split == 'test', 'smear'].nunique(), PAPER['test_smears']),
    'smears_in_both_splits': (int((smear_split > 1).sum()), 0),
}
for k, (got, exp) in checks.items():
    log(f'{"OK " if got == exp else "!! "}{k}: {got:,} (paper {exp:,})')

!! total: 103,674 (paper 103,675)
!! healthy: 34,720 (paper 34,721)
OK unhealthy: 2,942 (paper 2,942)
OK bothcells: 3,884 (paper 3,884)
!! rubbish: 62,128 (paper 62,074)
OK smears: 107 (paper 107)
OK test_smears: 20 (paper 20)
OK smears_in_both_splits: 0 (paper 0)


## Step 5 — Count cells per smear


In [6]:
sc = (man.pivot_table(index=['smear', 'split'], columns='cls', values='file',
                      aggfunc='count', fill_value=0)
        .reindex(columns=list(CLASSES), fill_value=0).reset_index())
sc['abnormal'] = sc['unhealthy'] + sc['bothcells']
sc['usable'] = sc['healthy'] + sc['abnormal']
sc = sc.sort_values(['split', 'abnormal'], ascending=[True, False])
sc.to_csv(os.path.join(OUT, 'smear_counts.csv'), index=False)
print('Test smears, most abnormal first:')
sc[sc.split == 'test'][['smear', 'healthy', 'abnormal', 'rubbish']]

Test smears, most abnormal first:


cls,smear,healthy,abnormal,rubbish
46,23699_16,188,165,2637
10,11363_16,1248,126,1256
92,669_15,842,98,1109
97,8361_12,114,78,856
59,26922_17,34,63,761
103,9413_12,191,59,522
76,30373_14,297,46,504
86,4038_18,253,44,290
91,6531_18,184,44,222
35,1913_18,199,42,641


## Step 6 — Propose the calibration split (17 of the 87 training smears)


In [7]:
train_pool = sc[sc.split == 'train'].sort_values(['abnormal', 'smear']).reset_index(drop=True)
N_CAL = 17
strata = np.array_split(np.arange(len(train_pool)), N_CAL)
rng = np.random.default_rng(SEED)
cal_idx = [int(rng.choice(s)) for s in strata]
train_pool['role'] = 'train'
train_pool.loc[cal_idx, 'role'] = 'calibration'
split_df = pd.concat([train_pool, sc[sc.split == 'test'].assign(role='test')], ignore_index=True)
split_df[['smear', 'role', 'healthy', 'abnormal', 'rubbish', 'usable']].to_csv(
    os.path.join(OUT, 'proposed_split.csv'), index=False)
print(split_df['role'].value_counts().to_string())

role
train          70
test           20
calibration    17


## Step 7 — What the counts mean for our design


In [8]:
def kish_eff_clusters(x):
    # Effective number of smears when cells are weighted by their count
    p = np.asarray(x, float); p = p[p > 0] / p.sum()
    return float(1 / (p ** 2).sum()) if len(p) else 0.0

def role_stats(df):
    A, H = int(df['abnormal'].sum()), int(df['healthy'].sum())
    return dict(smears=len(df), healthy=H, abnormal=A, usable=A + H,
                balanced_items=2 * min(A, H),
                smears_with_abnormal=int((df['abnormal'] > 0).sum()),
                top5_share_of_abnormal=round(float(df['abnormal'].nlargest(5).sum() / max(A, 1)), 3),
                eff_smears_abnormal=round(kish_eff_clusters(df['abnormal']), 1))

stats = {r: role_stats(split_df[split_df.role == r]) for r in ('train', 'calibration', 'test')}

N = stats['test']['balanced_items']
settings = [(0.55, 0.9), (0.55, 0.7), (0.9, 0.9), (0.9, 0.7)]
conflicts = {f'a={a},t={t}': dict(conflict_rate=round(a * (1 - t) + t * (1 - a), 3),
                                  expected_conflicts=round(N * (a * (1 - t) + t * (1 - a))),
                                  c=round(t * (1 - a) / (t * (1 - a) + a * (1 - t)), 3))
             for a, t in settings}

summary = dict(seed=SEED, checks={k: dict(got=g, paper=e) for k, (g, e) in checks.items()},
               roles=stats, test_conflicts=conflicts,
               note='Calibration split is PROPOSED; lock only after supervisor OK and OSF time-stamp.')
with open(os.path.join(OUT, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print('=== Table 1: per role ===')
print(pd.DataFrame(stats).T.to_string())
print('\n=== Table 2: expected conflicts on the balanced test set (target ~1,000) ===')
print(pd.DataFrame(conflicts).T.to_string())
print(f'\nSaved in {OUT}: osf_listing.csv, cells_manifest.csv, smear_counts.csv, proposed_split.csv, summary.json, count_log.txt')

=== Table 1: per role ===
             smears  healthy  abnormal   usable  balanced_items  smears_with_abnormal  top5_share_of_abnormal  eff_smears_abnormal
train          70.0  24735.0    4818.0  29553.0          9636.0                  66.0                   0.455                 17.2
calibration    17.0   4159.0     996.0   5155.0          1992.0                  17.0                   0.798                  5.7
test           20.0   5826.0    1012.0   6838.0          2024.0                  20.0                   0.524                 12.7

=== Table 2: expected conflicts on the balanced test set (target ~1,000) ===
              conflict_rate  expected_conflicts      c
a=0.55,t=0.9           0.46               931.0  0.880
a=0.55,t=0.7           0.48               972.0  0.656
a=0.9,t=0.9            0.18               364.0  0.500
a=0.9,t=0.7            0.34               688.0  0.206

Saved in /content/drive/MyDrive/CINC/data: osf_listing.csv, cells_manifest.csv, smear_counts.csv